# 03c — Create the MAG7 AI/BI Dashboard (OPTIONAL, pro mode)

Creates and **publishes** a Lakeview (AI/BI) dashboard over `ticker_data` for the pro
chatbot app's **Dashboard tab**. Published with embedding so it renders in the app
(`/embed/dashboardsv3/<id>`). The pro deploy notebook (`06b`) discovers it by name.

Prereqs: `00_tables_setup` (creates `ticker_data`). Run before `06b`.
Embedding also requires the workspace setting **Settings → Security → Embed dashboards →
Allow approved domains → `*.databricksapps.com`** (so the iframe isn't blocked by CSP).

In [ ]:
exec(open('../config.py').read())
import json
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me().user_name
# Pick a SQL warehouse (prefer serverless) unless sql_warehouse_id is pinned in config.py
wh = (sql_warehouse_id or "").strip()
if not wh:
    whs = list(w.warehouses.list())
    pick = next((x for x in whs if getattr(x,'enable_serverless_compute',False)), (whs or [None])[0])
    wh = getattr(pick,'id','')
assert wh, "No SQL warehouse found; set sql_warehouse_id in config.py"
T = f"{catalog}.{schema}.ticker_data"
print("warehouse:", wh, "| table:", T)

In [ ]:
DASH_NAME = "MAG7 Market Dashboard"
ds_latest = ("SELECT company_name, CAST(market_cap AS DOUBLE) AS market_cap, "
  "CAST(beta AS DOUBLE) AS beta, CAST(pe_trailing AS DOUBLE) AS pe_trailing, "
  "CAST(pe_forward AS DOUBLE) AS pe_forward, CAST(ev_ebitda AS DOUBLE) AS ev_ebitda "
  f"FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY company_name ORDER BY Date DESC) AS rn FROM {T}) WHERE rn = 1")
ds_ts = (f"SELECT Date AS trade_date, company_name, CAST(price_close AS DOUBLE) AS price_close, "
  f"CAST(volume AS DOUBLE) AS volume FROM {T}")

def text(n,m,y): return {"widget":{"name":n,"multilineTextboxSpec":{"lines":[m]}},"position":{"x":0,"y":y,"width":6,"height":1}}
bar={"widget":{"name":"mktcap_bar","queries":[{"name":"main_query","query":{"datasetName":"ds_latest","fields":[{"name":"company_name","expression":"`company_name`"},{"name":"sum(market_cap)","expression":"SUM(`market_cap`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"bar","encodings":{"x":{"fieldName":"company_name","scale":{"type":"categorical","sort":{"by":"y-reversed"}},"displayName":"Company"},"y":{"fieldName":"sum(market_cap)","scale":{"type":"quantitative"},"displayName":"Market cap"},"label":{"show":True}},"frame":{"showTitle":True,"title":"Latest market cap by company"},"mark":{"colors":["#FF3621","#FFAB00","#00A972","#8BCAE7","#AB4057","#99DDB4","#FCA4A1"]}}},"position":{"x":0,"y":2,"width":3,"height":6}}
line={"widget":{"name":"close_line","queries":[{"name":"main_query","query":{"datasetName":"ds_ts","fields":[{"name":"daily(trade_date)","expression":"DATE_TRUNC(\"DAY\", `trade_date`)"},{"name":"company_name","expression":"`company_name`"},{"name":"avg(price_close)","expression":"AVG(`price_close`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"line","encodings":{"x":{"fieldName":"daily(trade_date)","scale":{"type":"temporal"},"displayName":"Date"},"y":{"fieldName":"avg(price_close)","scale":{"type":"quantitative"},"displayName":"Close price"},"color":{"fieldName":"company_name","scale":{"type":"categorical"},"displayName":"Company"}},"frame":{"showTitle":True,"title":"Close price over time"}}},"position":{"x":3,"y":2,"width":3,"height":6}}
area={"widget":{"name":"vol_area","queries":[{"name":"main_query","query":{"datasetName":"ds_ts","fields":[{"name":"daily(trade_date)","expression":"DATE_TRUNC(\"DAY\", `trade_date`)"},{"name":"company_name","expression":"`company_name`"},{"name":"sum(volume)","expression":"SUM(`volume`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"area","encodings":{"x":{"fieldName":"daily(trade_date)","scale":{"type":"temporal"},"displayName":"Date"},"y":{"fieldName":"sum(volume)","scale":{"type":"quantitative"},"displayName":"Volume"},"color":{"fieldName":"company_name","scale":{"type":"categorical"},"displayName":"Company"}},"frame":{"showTitle":True,"title":"Trading volume over time"}}},"position":{"x":0,"y":8,"width":6,"height":5}}
table={"widget":{"name":"fundamentals_tbl","queries":[{"name":"main_query","query":{"datasetName":"ds_latest","fields":[{"name":"company_name","expression":"`company_name`"},{"name":"market_cap","expression":"`market_cap`"},{"name":"beta","expression":"`beta`"},{"name":"pe_trailing","expression":"`pe_trailing`"},{"name":"pe_forward","expression":"`pe_forward`"},{"name":"ev_ebitda","expression":"`ev_ebitda`"}],"disaggregated":True}}],"spec":{"version":2,"widgetType":"table","encodings":{"columns":[{"fieldName":"company_name","displayName":"Company"},{"fieldName":"market_cap","displayName":"Market cap"},{"fieldName":"beta","displayName":"Beta"},{"fieldName":"pe_trailing","displayName":"P/E (trailing)"},{"fieldName":"pe_forward","displayName":"P/E (forward)"},{"fieldName":"ev_ebitda","displayName":"EV/EBITDA"}]},"frame":{"showTitle":True,"title":"Latest fundamentals"}}},"position":{"x":0,"y":13,"width":6,"height":6}}
dash={"datasets":[{"name":"ds_latest","displayName":"Latest per company","queryLines":[ds_latest]},{"name":"ds_ts","displayName":"Daily ticker time series","queryLines":[ds_ts]}],"pages":[{"name":"overview","displayName":"MAG7 Overview","pageType":"PAGE_TYPE_CANVAS","layout":[text("title","## MAG7 Market Dashboard",0),text("subtitle","Fundamentals and price/volume trends for the Magnificent 7, from ticker_data.",1),bar,line,area,table]}],"uiSettings":{"applyModeEnabled":False}}

# Create (or reuse) the dashboard, then publish with embedding.
existing = next((d for d in w.api_client.do("GET","/api/2.0/lakeview/dashboards").get("dashboards",[])
                 if d.get("display_name")==DASH_NAME and d.get("lifecycle_state")!="TRASHED"), None)
if existing:
    did = existing["dashboard_id"]
    w.api_client.do("PATCH", f"/api/2.0/lakeview/dashboards/{did}", body={"serialized_dashboard":json.dumps(dash)})
    print("Updated existing dashboard:", did)
else:
    res = w.api_client.do("POST","/api/2.0/lakeview/dashboards", body={"display_name":DASH_NAME,"warehouse_id":wh,"parent_path":f"/Users/{me}","serialized_dashboard":json.dumps(dash)})
    did = res["dashboard_id"]; print("Created dashboard:", did)

w.api_client.do("POST", f"/api/2.0/lakeview/dashboards/{did}/published", body={"embed_credentials":True,"warehouse_id":wh})
print(f"✅ Published MAG7 dashboard (embed enabled). dashboard_id = {did}")
print("06b discovers it by name; or set aibi_dashboard_id in config.py.")